In [ ]:
!pip install nltk # using the Python APIs of NLTK instead of the Stanford NLP toolsuite in Java!
from nltk import download, sent_tokenize, word_tokenize
#download('punkt_tab')
#download('wordnet')

from nltk.stem import WordNetLemmatizer
import math

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("RunLSA").getOrCreate()
sc = spark.sparkContext

In [3]:
# Basic hyper-parameters

sampleSize = 0.01  # 1 percent of available files. change to 1.0 for full experiment
numTerms = 5000    # change to 50000 for full experiment
k = 12             # number of latent concepts in the reduced matrix

In [4]:
# Parse the headers of the Wikipedia articles

def parseHeader(line):
    try:
        s = line[line.index("id=\"") + 4:]
        article_id = s[:s.index("\"")]
        s = s[s.index("url=\"") + 5:]
        url = s[:s.index("\"")]
        s = s[s.index("title=\"") + 7:]
        title = s[:s.index("\"")]
        return article_id, url, title
    except Exception as e:
        return "", "", ""

def parse(lines):
    docs = []
    title = ""
    content = ""
    for line in lines:
        try:
            if line.startswith("<doc "):
                title = parseHeader(line)[2]
                content = ""
            elif line.startswith("</doc>"):
                if title and content:
                    docs.append((title, content))
            else:
                content += line + "\n"
        except Exception as e:
            content = ""
    return docs

In [5]:
# Read the Wiki files as whole text files!

textFiles = sc.wholeTextFiles("../Data/Wikipedia-En-41784-Articles/*/*").sample(False, sampleSize)
numFiles = textFiles.count()

print("Number of Wiki files:", numFiles) 

[Stage 0:>                                                          (0 + 2) / 2]

Number of Wiki files: 12


In [6]:
# Parse & flatMap the input text

def parseFlatMap(uri_text):
    uri, text = uri_text
    return parse(text.split("\n"))

plainText = textFiles.flatMap(parseFlatMap)
plainText.cache()

numDocs = plainText.count()
bNumDocs = sc.broadcast(numDocs)

print("Number of docs in Wiki files:", numDocs)

[Stage 1:>                                                          (0 + 2) / 2]

Number of docs in Wiki files: 427


In [7]:
# Parse & broadcast stopwords

stopwords = set(sc.textFile("../Data/stopwords.txt").collect())
bStopWords = sc.broadcast(stopwords)

print("Number of stopwords:", len(bStopWords.value))

Number of stopwords: 214


In [8]:
# Define NLP pipeline

def plainTextToLemmas(title_text):
    title, text = title_text
    lemmatizer = WordNetLemmatizer()
    lemmas = []
    sentences = sent_tokenize(text) # sentence splitting    
    for sentence in sentences:
        tokens = word_tokenize(sentence) # tokenization            
        for token in tokens:
            lemma = lemmatizer.lemmatize(token.lower()) # lemmatization
            if len(lemma) > 2 and lemma not in bStopWords.value and lemma.isalpha():
                lemmas.append(lemma)
    return (title, lemmas)

def plainTextToLemmas2(title_text): # simpler alternative text splitter
    title, text = title_text
    import re
    return (title, [w.lower() for w in re.split(r'\W+', text)])

# Call NLP pipeline, once per partition
lemmatized = plainText.mapPartitions(lambda x: map(plainTextToLemmas, x))
#lemmatized.first() # print sample

In [9]:
# Calculate term frequencies (TF)

def calculateTermFreqs(title_terms):
    title, terms = title_terms
    termFreqs = {}
    for term in terms:
        termFreqs[term] = termFreqs.get(term, 0) + 1
    return (title, termFreqs)

docTermFreqs = lemmatized.map(calculateTermFreqs)
docTermFreqs.cache()
#docTermFreqs.first() # print sample

PythonRDD[7] at RDD at PythonRDD.scala:53

In [10]:
# Extract document IDs and zip with unique IDs
docIds = docTermFreqs.map(lambda x: x[0]).zipWithUniqueId().map(lambda x: (x[1], x[0])).collectAsMap()

# Aggregate document frequencies
docFreqs = docTermFreqs.flatMap(lambda x: x[1].keys()).map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y, numPartitions=24)

ordering = lambda x: x[1]
topDocFreqs = docFreqs.top(numTerms, key=ordering) # sort by frequency

# Calculate inverse document frequencies (IDF)
idfs = {term: math.log(numDocs / count) for term, count in topDocFreqs}
idTerms = dict(zip(idfs.keys(), range(len(idfs)))) # term IDs forward mapping
termIds = {v: k for k, v in idTerms.items()} # term IDs backward mapping

bIdfs = sc.broadcast(idfs) # broadcast IDF values
bIdTerms = sc.broadcast(idTerms) # broadcast term mappings (needed only in forward direction)

In [11]:
import numpy as np
from scipy.sparse import csr_matrix

from pyspark.mllib.linalg import Vectors, Matrices
from pyspark.mllib.linalg.distributed import RowMatrix, SingularValueDecomposition

In [12]:
# This part constructs the row vectors from the term frequencies (TF) and the inverse document frequencies (IDF)

rowVectors = docTermFreqs.map(lambda x: x[1]).map(
    lambda termFreqs:
        Vectors.sparse(len(bIdTerms.value), 
            [(bIdTerms.value[term], bIdfs.value[term] * termFreqs[term] / sum(termFreqs.values())) 
                for term in termFreqs.keys() if term in bIdTerms.value]))

rowVectors.cache()

# And finally compute the SVD
mat = RowMatrix(rowVectors) 
svd = mat.computeSVD(k, computeU=True)

25/04/10 14:11:49 WARN RowMatrix: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.
25/04/10 14:11:49 WARN InstanceBuilder$NativeARPACK: Failed to load implementation from:dev.ludovic.netlib.arpack.JNIARPACK
25/04/10 14:11:56 WARN RowMatrix: The input data was not directly cached, which may hurt performance if its parent RDDs are also uncached.


In [13]:
# Helper functions to query the latent semantic index

def topTermsInTopConcepts(svd: SingularValueDecomposition, numConcepts, numTerms):
    
    topTerms = []
    v = svd.V
    arr = v.toArray().T
    for i in range(numConcepts):
        termWeights = [(arr[i][termId], termId) for termId in range(v.numRows)]
        sorted_terms = sorted(termWeights, key=lambda x: -x[0])
        topTerms.append([(termIds.get(idx, str(idx)), score) for score, idx in sorted_terms[:numTerms]])
    
    return topTerms

def topDocsInTopConcepts(svd: SingularValueDecomposition, numConcepts, numDocs):

    topDocs = []
    u = svd.U
    for i in range(numConcepts):
        docWeights = [(score, docId) for score, docId in u.rows.map(lambda row: row.toArray()[i]).zipWithUniqueId().collect()]
        sorted_docs = sorted(docWeights, key=lambda x: -x[0])
        topDocs.append([(docIds.get(idx, str(idx)), score) for score, idx in sorted_docs[:numDocs]])
    
    return topDocs

top_concept_terms = topTermsInTopConcepts(svd, 12, 12)
top_concept_docs = topDocsInTopConcepts(svd, 12, 12)

for terms, docs in zip(top_concept_terms, top_concept_docs):
    print("Concept terms: " + ", ".join([term for term, _ in terms]))
    print("Concept docs: " + ", ".join([doc for doc, _ in docs]))
    print()

Concept terms: promptly, talented, otl, authored, albeit, shutout, rallied, treason, alex, disagreed, marched, jeff
Concept docs: 1630s BC, 1640s BC, 1690s BC, 1680s BC, 1670s BC, 1660s BC, 1610s BC, 1620s BC, 1650s BC, Daisy cutter, Alfonso XI of Castile, Matthew George Easton

Concept terms: refer, storage, virtual, bond, may, terminal, agent, craig, daniel, music, thing, view
Concept docs: Virtual storage, Storage, BBS, Terminal, Bonds, Agent, Bond, Virtual call capability, Madness, Modulo, Booze, Validation

Concept terms: storage, virtual, capability, call, telecommunication, facility, feature, sometimes, service, circuit, protocol, layer
Concept docs: Virtual storage, Storage, Virtual call capability, Virtual circuit, Virtual terminal, Variable-length buffer, Voice-operated switch, Hemolysis, Daminozide, User information bit, Electron, Wideband modem

Concept terms: decade, game, singapore, publisher, comic, printing, roman, forest, jean, aesthetic, poetry, team
Concept docs: 160

In [14]:
# Running keyword queries

def termsToQueryVector(terms, idTerms, idfs):
    indices = [idTerms[term] for term in terms]
    values = [idfs[term] for term in terms]
    return csr_matrix((values, (indices, [0]*len(indices))), shape=(len(idTerms), 1))

def topDocsForTermQuery(US, V, query):
    term_row_arr = np.dot(V.toArray().T, query.toarray()).flatten()
    term_row_vec = Matrices.dense(len(term_row_arr), 1, term_row_arr)
    doc_scores = US.multiply(term_row_vec)
    all_doc_weights = doc_scores.rows.zipWithUniqueId().map(lambda x: (x[0].toArray()[0], x[1]))
    return sorted(all_doc_weights.collect(), key=lambda x: -x[0])[:10]

def multiplyByDiagonalRowMatrix(mat, diag):
    s_arr = diag.toArray()
    return RowMatrix(mat.rows.map(lambda vec: Vectors.dense(np.multiply(vec.toArray(), s_arr))))

US = multiplyByDiagonalRowMatrix(svd.U, svd.s)

terms = ["serious", "incident"]
queryVec = termsToQueryVector(terms, idTerms, idfs)
topDocsForTermQuery(US, svd.V, queryVec)

[(np.float64(0.0003779381317112024), 25),
 (np.float64(0.000371230535027015), 147),
 (np.float64(0.000371230535027015), 149),
 (np.float64(0.00033699397571512713), 365),
 (np.float64(0.0003356737238773748), 369),
 (np.float64(0.00033052118205565795), 371),
 (np.float64(0.0002965870653013009), 151),
 (np.float64(0.0002051357356747645), 363),
 (np.float64(0.00019588096664781754), 29),
 (np.float64(0.0001766860879412005), 375)]

In [15]:
# Dimensions of matrix UxS
num_rows_US = US.numRows()
num_cols_US = US.numCols()

# Dimensions of matrix V
num_rows_V = svd.V.numRows
num_cols_V = svd.V.numCols

print("Dimensions of matrix UxS:")
print(f"Number of rows: {num_rows_US}")
print(f"Number of columns: {num_cols_US}")

print("\nDimensions of matrix V:")
print(f"Number of rows: {num_rows_V}")
print(f"Number of columns: {num_cols_V}")

Dimensions of matrix UxS:
Number of rows: 427
Number of columns: 12

Dimensions of matrix V:
Number of rows: 5000
Number of columns: 12
